# Table 5: Shape-Specific Regularized Sigma-Star Results

## Step 1: Run shape-specific comparison

This step computes Raw-18, global sigma-star, and regularized temperature-dependent sigma-star results for each dataset and specimen shape using 5-fold cross-validation.


In [1]:
"""
All materials: shape-specific correction-regularized sigma*_T-18.

For each material/family and each specimen shape, compare:
    Raw-18        [stress, T, composition]
    Global-18     [sigma*, T, composition]
    Tdep-18       [sigma*_T, T, composition]
    Reg-Tdep-18   [regularized sigma*_T, T, composition]
"""

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

try:
    from numpy.exceptions import RankWarning
except ImportError:
    RankWarning = np.RankWarning
warnings.simplefilter("ignore", RankWarning)


DATA_PATH = os.path.join("..", "Dataset", "Creep_Properties_Dataset.xlsx")

COMP_FEATURES = [
    "C, wt.%",
    "Si, wt.%",
    "Mn, wt.%",
    "P, wt.%",
    "S, wt.%",
    "Ni, wt.%",
    "Cr, wt.%",
    "Mo, wt.%",
    "Al, wt.%",
    "N, wt.%",
    "Ti, wt.%",
    "Fe, wt.%",
    "Nb, wt.%",
    "B, wt.%",
    "Cu, wt.%",
    "Co, wt.%",
]
GEO_FEATURES = ["Thickness (mm)", "Cross-Sectional Area", "Aspect Ratio (L/D)"]
BASE18_FEATURES = ["Stress (Mpa)", "Temperature of Creep Test (C)"] + COMP_FEATURES
BASE21_FEATURES = BASE18_FEATURES + GEO_FEATURES

GBM_PARAMS = dict(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    min_samples_leaf=3,
    random_state=42,
)
GBM_PARAMS_SMALL = dict(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=3,
    random_state=42,
)
N_SPLITS = 5
MIN_N = 20
LAMBDAS = [0.0, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
SHAPES = ["Flat", "Cylindrical"]

DATASETS = {
    "SS316H": {"alloys": ["SS316H"], "sigma_params": 3},
    "SS304H": {"alloys": ["SS304H"], "sigma_params": 3},
    "SS316L": {"alloys": ["SS316L"], "sigma_params": 3},
    "SA533B": {"alloys": ["SA533B"], "sigma_params": 2},
    "SS316 Family": {"alloys": ["SS316H", "SS316L", "SS316LN", "SS316FR"], "sigma_params": 3},
    "SS304 Family": {"alloys": ["SS304H", "SS304J", "SS304L"], "sigma_params": 3},
    "RPV Family": {"alloys": ["SA533B", "SA508", "A508/A533"], "sigma_params": 2},
    "SS316LN": {"alloys": ["SS316LN"], "sigma_params": 3},
    "SS316FR": {"alloys": ["SS316FR"], "sigma_params": 3},
    "SS304J": {"alloys": ["SS304J"], "sigma_params": 3},
    "SS304L": {"alloys": ["SS304L"], "sigma_params": 3},
    "A709": {"alloys": ["A709"], "sigma_params": 3},
    "IN-617": {"alloys": ["IN-617"], "sigma_params": 3},
    "IN-600": {"alloys": ["IN-600"], "sigma_params": 3},
    "Zr-3": {"alloys": ["Zr-3"], "sigma_params": 3},
    "SA508": {"alloys": ["SA508"], "sigma_params": 2},
    "A508/A533": {"alloys": ["A508/A533"], "sigma_params": 2},
}

RPV_ALLOYS = {"SA533B", "SA508", "A508/A533"}


def impute_rpv_cross_section(df: pd.DataFrame) -> pd.DataFrame:
    """RPV Cross-Sectional Area."""
    df = df.copy()
    rpv_mask = df["Alloy Designation"].isin(RPV_ALLOYS)
    missing = rpv_mask & df["Cross-Sectional Area"].isna()

    cyl = missing & (df["Specimen Shape"] == "Cylindrical")
    d = df.loc[cyl, "Thickness (mm)"]
    df.loc[cyl, "Cross-Sectional Area"] = np.pi * (d / 2) ** 2
    n_cyl = int(cyl.sum())

    flat = missing & (df["Specimen Shape"] == "Flat")
    rpv_median_overall = df.loc[
        rpv_mask & df["Cross-Sectional Area"].notna(), "Cross-Sectional Area"
    ].median()
    n_flat = 0
    for alloy in RPV_ALLOYS:
        alloy_flat = flat & (df["Alloy Designation"] == alloy)
        if not alloy_flat.any():
            continue
        alloy_median = df.loc[
            rpv_mask
            & (df["Alloy Designation"] == alloy)
            & df["Cross-Sectional Area"].notna(),
            "Cross-Sectional Area",
        ].median()
        fill_val = alloy_median if not np.isnan(alloy_median) else rpv_median_overall
        df.loc[alloy_flat, "Cross-Sectional Area"] = fill_val
        n_flat += int(alloy_flat.sum())

    print(
        f"  [RPV] imputed Cross-Sectional Area: "
        f"{n_cyl} cylindrical (formula), {n_flat} flat (median)"
    )
    return df


def impute_ss316l_composition(df: pd.DataFrame) -> pd.DataFrame:
    """SS316L composition."""
    df = df.copy()
    mask = df["Alloy Designation"] == "SS316L"
    comp_missing = mask & df[COMP_FEATURES[0]].isna()
    if not comp_missing.any():
        return df

    medians = df.loc[mask & ~comp_missing, COMP_FEATURES].median()
    for col in COMP_FEATURES:
        df.loc[comp_missing, col] = medians[col]
    print(
        f"  [SS316L] imputed composition for {int(comp_missing.sum())} records "
        f"using SS316L per-column medians"
    )
    return df


def metrics(y_true, y_pred):
    return (
        float(r2_score(y_true, y_pred)),
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        float(np.mean(np.abs(y_true - y_pred) < np.log10(2)) * 100.0),
    )


def proxy(ls_star, y, T):
    total, n = 0.0, 0
    for tv in np.unique(T):
        mk = T == tv
        if int(np.sum(mk)) < 3:
            continue
        if np.std(ls_star[mk]) < 1e-10:
            resid = y[mk] - np.mean(y[mk])
        else:
            try:
                coef = np.polyfit(ls_star[mk], y[mk], 1)
                resid = y[mk] - np.polyval(coef, ls_star[mk])
            except np.linalg.LinAlgError:
                resid = y[mk] - np.mean(y[mk])
        total += float(np.var(resid)) * int(np.sum(mk))
        n += int(np.sum(mk))
    return total / max(n, 1)


def apply_global(p, n_params, ls, lb, lA, lLD):
    if n_params == 3:
        out = ls + p[0] * lb + p[1] * lA + p[2] * lLD
    else:
        out = ls + p[0] * lb + p[1] * lLD
    return np.clip(out, -1.0, 4.0)


def apply_tdep(p, n_params, ls, Tn, lb, lA, lLD):
    if n_params == 3:
        out = (
            ls
            + (p[0] + p[3] * Tn) * lb
            + (p[1] + p[4] * Tn) * lA
            + (p[2] + p[5] * Tn) * lLD
        )
    else:
        out = ls + (p[0] + p[2] * Tn) * lb + (p[1] + p[3] * Tn) * lLD
    return np.clip(out, -1.0, 4.0)


def opt_global(n_params, ls, y, T, lb, lA, lLD):
    x0 = [0.0, 0.0, -0.1] if n_params == 3 else [0.0, -0.1]
    res = minimize(
        lambda p: proxy(apply_global(p, n_params, ls, lb, lA, lLD), y, T),
        x0,
        method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-5, "fatol": 1e-6},
    )
    return res.x


def opt_tdep_regularized(n_params, ls, y, T, Tn, lb, lA, lLD, p0, lam):
    if n_params == 3:
        x0 = [p0[0], p0[1], p0[2], 0.0, 0.0, 0.0]
        maxiter = 5000
    else:
        x0 = [p0[0], p0[1], 0.0, 0.0]
        maxiter = 4000

    def objective(p):
        ls_star = apply_tdep(p, n_params, ls, Tn, lb, lA, lLD)
        return proxy(ls_star, y, T) + lam * float(np.mean((ls_star - ls) ** 2))

    res = minimize(
        objective,
        x0,
        method="Nelder-Mead",
        options={"maxiter": maxiter, "xatol": 1e-5, "fatol": 1e-6},
    )
    return res.x


def fit_predict(X, y, tr, val, gbm_params):
    model = GradientBoostingRegressor(**gbm_params)
    model.fit(X[tr], y[tr])
    return model.predict(X[val])


def replace_stress(X, ls_star):
    out = X.copy()
    out[:, 0] = 10.0**ls_star
    return out


def skipped_rows(ds_name, shape, alloys, n, n_params, status):
    return [
        {
            "dataset": ds_name,
            "shape": shape,
            "alloys": ",".join(alloys),
            "n": n,
            "sigma_params": n_params,
            "gbm": "skipped",
            "status": status,
            "model": "skipped",
            "r2": np.nan,
            "rmse": np.nan,
            "within_2x_pct": np.nan,
            "delta_vs_raw18": np.nan,
        }
    ]


print("Loading data ...")
df = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)
df = impute_rpv_cross_section(df)
df = impute_ss316l_composition(df)
all_rows = []

base_mask = (
    (df["Notch"] == "unnotched")
    & df["Creep Rupture Life (h)"].notna()
    & (df["Creep Rupture Life (h)"] > 0)
    & (df["Stress (Mpa)"] > 0)
    & df["Specimen Shape"].isin(SHAPES)
)
base = df[base_mask].copy().reset_index(drop=True)
base["log_tR"] = np.log10(base["Creep Rupture Life (h)"])
base["log_sigma"] = np.log10(base["Stress (Mpa)"])

for ds_name, cfg in DATASETS.items():
    alloys = cfg["alloys"]
    n_params = cfg["sigma_params"]
    print(f"\n{'=' * 92}\nDataset: {ds_name}")

    data = base[base["Alloy Designation"].isin(alloys)].copy().reset_index(drop=True)
    extra_cols = []
    for alloy in alloys[1:]:
        col = f"is_{alloy.replace('/', '_')}"
        data[col] = (data["Alloy Designation"] == alloy).astype(float)
        extra_cols.append(col)

    feat18 = BASE18_FEATURES + extra_cols
    needed = BASE21_FEATURES + extra_cols + ["log_tR", "log_sigma", "Specimen Shape"]

    for shape in SHAPES:
        sub = data[data["Specimen Shape"] == shape].copy()
        sub = sub[needed].dropna(subset=BASE21_FEATURES + ["log_tR"]).reset_index(drop=True)
        n = len(sub)

        if n < MIN_N:
            print(f"  {shape:<12} skipped: n={n} < {MIN_N}")
            all_rows.extend(skipped_rows(ds_name, shape, alloys, n, n_params, "n_too_small"))
            continue

        y = sub["log_tR"].values.astype(float)
        ls = sub["log_sigma"].values.astype(float)
        T = sub["Temperature of Creep Test (C)"].values.astype(float)
        Tn = (T - float(T.mean())) / max(float(T.std()), 1e-9)

        b = sub["Thickness (mm)"].values.astype(float)
        A = sub["Cross-Sectional Area"].values.astype(float)
        LD = sub["Aspect Ratio (L/D)"].values.astype(float)
        b0 = float(np.nanmedian(b))
        A0 = float(np.nanmedian(A))
        LD0 = float(np.nanmedian(LD))
        lb = np.log10(np.where(b > 0, b / b0, 1.0))
        lA = np.log10(np.where(A > 0, A / A0, 1.0))
        lLD = np.log10(np.where(LD > 0, LD / LD0, 1.0))

        X18 = sub[feat18].values.astype(float)
        gbm_params = GBM_PARAMS_SMALL if n < 150 else GBM_PARAMS
        gbm_kind = "small" if n < 150 else "full"

        preds = {
            "Raw-18": np.zeros(n),
            "Star-global-18": np.zeros(n),
        }
        for lam in LAMBDAS:
            preds[f"Reg-Tdep-18 lambda={lam:g}"] = np.zeros(n)

        print(f"  {shape:<12} n={n}, gbm={gbm_kind}")
        kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
        for fold, (tr, val) in enumerate(kf.split(X18), 1):
            preds["Raw-18"][val] = fit_predict(X18, y, tr, val, gbm_params)

            p_global = opt_global(n_params, ls[tr], y[tr], T[tr], lb[tr], lA[tr], lLD[tr])
            ls_global = apply_global(p_global, n_params, ls, lb, lA, lLD)
            preds["Star-global-18"][val] = fit_predict(
                replace_stress(X18, ls_global), y, tr, val, gbm_params
            )

            for lam in LAMBDAS:
                p = opt_tdep_regularized(
                    n_params, ls[tr], y[tr], T[tr], Tn[tr], lb[tr], lA[tr], lLD[tr], p_global, lam
                )
                ls_star = apply_tdep(p, n_params, ls, Tn, lb, lA, lLD)
                preds[f"Reg-Tdep-18 lambda={lam:g}"][val] = fit_predict(
                    replace_stress(X18, ls_star), y, tr, val, gbm_params
                )

        raw_r2 = metrics(y, preds["Raw-18"])[0]
        for model, pred in preds.items():
            r2, rmse, w2 = metrics(y, pred)
            all_rows.append(
                {
                    "dataset": ds_name,
                    "shape": shape,
                    "alloys": ",".join(alloys),
                    "n": n,
                    "sigma_params": n_params,
                    "gbm": gbm_kind,
                    "status": "ok",
                    "model": model,
                    "r2": r2,
                    "rmse": rmse,
                    "within_2x_pct": w2,
                    "delta_vs_raw18": r2 - raw_r2,
                }
            )

        best_name = max(
            [m for m in preds if m.startswith("Reg-Tdep-18")],
            key=lambda m: metrics(y, preds[m])[0],
        )
        print(f"    Raw18={raw_r2:.4f}, best={best_name}, R2={metrics(y, preds[best_name])[0]:.4f}")

long_df = pd.DataFrame(all_rows)

summary_rows = []
for (dataset, shape), grp in long_df.groupby(["dataset", "shape"], sort=False):
    first = grp.iloc[0]
    if first["status"] != "ok":
        summary_rows.append(
            {
                "dataset": dataset,
                "shape": shape,
                "alloys": first["alloys"],
                "n": first["n"],
                "gbm": first["gbm"],
                "status": first["status"],
                "raw18_r2": np.nan,
                "global18_r2": np.nan,
                "tdep18_unregularized_r2": np.nan,
                "best_regularized_model": "skipped",
                "best_regularized_r2": np.nan,
                "best_delta_vs_raw18": np.nan,
                "extra_vs_unregularized": np.nan,
                "best_regularized_rmse": np.nan,
                "best_regularized_w2x": np.nan,
            }
        )
        continue

    raw = grp[grp["model"] == "Raw-18"].iloc[0]
    global18 = grp[grp["model"] == "Star-global-18"].iloc[0]
    tdep0 = grp[grp["model"] == "Reg-Tdep-18 lambda=0"].iloc[0]
    reg = grp[grp["model"].str.startswith("Reg-Tdep-18")]
    best = reg.loc[reg["r2"].idxmax()]
    summary_rows.append(
        {
            "dataset": dataset,
            "shape": shape,
            "alloys": first["alloys"],
            "n": first["n"],
            "gbm": first["gbm"],
            "status": first["status"],
            "raw18_r2": raw["r2"],
            "global18_r2": global18["r2"],
            "tdep18_unregularized_r2": tdep0["r2"],
            "best_regularized_model": best["model"],
            "best_regularized_r2": best["r2"],
            "best_delta_vs_raw18": best["r2"] - raw["r2"],
            "extra_vs_unregularized": best["r2"] - tdep0["r2"],
            "best_regularized_rmse": best["rmse"],
            "best_regularized_w2x": best["within_2x_pct"],
        }
    )

summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 142)
print(
    f"{'Dataset':<16} {'Shape':<12} {'n':>5} {'Raw18':>8} {'Global':>8} "
    f"{'Tdep0':>8} {'BestReg':>8} {'Delta':>8} {'Extra':>8} {'Best model':<26} {'Status':<10}"
)
print("-" * 142)
for _, r in summary.iterrows():
    if r["status"] != "ok":
        print(f"{r['dataset']:<16} {r['shape']:<12} {int(r['n']):>5} {'':>8} {'':>8} {'':>8} {'':>8} {'':>8} {'':>8} {'skipped':<26} {r['status']:<10}")
        continue
    print(
        f"{r['dataset']:<16} {r['shape']:<12} {int(r['n']):>5} "
        f"{r['raw18_r2']:>8.4f} {r['global18_r2']:>8.4f} "
        f"{r['tdep18_unregularized_r2']:>8.4f} {r['best_regularized_r2']:>8.4f} "
        f"{r['best_delta_vs_raw18']:>+8.4f} {r['extra_vs_unregularized']:>+8.4f} "
        f"{r['best_regularized_model']:<26} {r['status']:<10}"
    )
print("=" * 142)


Loading data ...
  [RPV] imputed Cross-Sectional Area: 15 cylindrical (formula), 47 flat (median)
  [SS316L] imputed composition for 15 records using SS316L per-column medians

Dataset: SS316H
  Flat         n=377, gbm=full
    Raw18=0.9614, best=Reg-Tdep-18 lambda=0.3, R2=0.9655
  Cylindrical  n=689, gbm=full
    Raw18=0.9185, best=Reg-Tdep-18 lambda=0.01, R2=0.9277

Dataset: SS304H
  Flat         n=289, gbm=full
    Raw18=0.9665, best=Reg-Tdep-18 lambda=0, R2=0.9665
  Cylindrical  n=545, gbm=full
    Raw18=0.8370, best=Reg-Tdep-18 lambda=0.001, R2=0.8384

Dataset: SS316L
  Flat         skipped: n=0 < 20
  Cylindrical  n=120, gbm=small
    Raw18=0.8351, best=Reg-Tdep-18 lambda=1, R2=0.8429

Dataset: SA533B
  Flat         n=31, gbm=small
    Raw18=-0.0455, best=Reg-Tdep-18 lambda=0, R2=0.0739
  Cylindrical  n=35, gbm=small
    Raw18=0.3369, best=Reg-Tdep-18 lambda=0, R2=0.3369

Dataset: SS316 Family
  Flat         n=377, gbm=full
    Raw18=0.9612, best=Reg-Tdep-18 lambda=0.3, R2=0.9653

## Final Table 5

This step selects the five paper rows from the Step 1 summary and formats the paper-ready Table 5.


In [2]:
ROW_ORDER = [
    ("SS316H", "Flat"),
    ("SS316H", "Cylindrical"),
    ("SS304H", "Flat"),
    ("SS304H", "Cylindrical"),
    ("SS316L", "Cylindrical"),
]

rows = []
for dataset, shape in ROW_ORDER:
    match = summary[(summary["dataset"] == dataset) & (summary["shape"] == shape)]
    if len(match) != 1:
        raise ValueError(f"Expected one row for {dataset} / {shape}, found {len(match)}")
    row = match.iloc[0]
    rows.append(
        {
            "Dataset": dataset,
            "Shape": shape,
            "Number of data points": int(row["n"]),
            "Nominal stress sigma": f"{row['raw18_r2']:.4f}",
            "Regularized temperature-dependent geometry-corrected stress sigma*_T": (
                f"{row['best_regularized_r2']:.4f}"
            ),
        }
    )

final_table = pd.DataFrame(rows)
final_table

,Dataset,Shape,Number of data points,Nominal stress sigma,Regularized temperature-dependent geometry-corrected stress sigma*_T
0,SS316H,Flat,377,0.9614,0.9655
1,SS316H,Cylindrical,689,0.9185,0.9277
2,SS304H,Flat,289,0.9665,0.9665
3,SS304H,Cylindrical,545,0.8370,0.8384
4,SS316L,Cylindrical,120,0.8351,0.8429
